# 02 - Embedding Steps Overview
Notebook nay giup ban theo doi quy trinh embed/extract o muc workflow.

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
host_dir = ROOT / 'data' / 'host_images'
wm_path = ROOT / 'data' / 'watermark' / 'cict.png'
results_dir = ROOT / 'results'

print('Root:', ROOT)
print('Host dir exists:', host_dir.exists())
print('Watermark exists:', wm_path.exists())
print('Results dir exists:', results_dir.exists())

## Step map
1. Chon host image va watermark.
2. Embed watermark vao ROI theo pipeline SWT + DCT + QIM.
3. Tao cac attacked images.
4. Recover watermark va tinh metrics.

Ban co the chay pipeline truoc bang lenh:
`python scripts/main.py --help` de xem tham so, sau do chay theo kich ban mong muon.

In [ ]:
host_images = sorted([p for p in host_dir.glob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp'}])
if not host_images:
    raise FileNotFoundError('Khong tim thay host image trong data/host_images.')

host_path = host_images[0]
host_name = host_path.stem
attack_dir = results_dir / host_name / 'attack'
recover_dir = results_dir / host_name / 'recover'

print('Selected host:', host_path.name)
print('Expected attack dir:', attack_dir)
print('Expected recover dir:', recover_dir)

In [ ]:
def read_rgb(path: Path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

host_img = read_rgb(host_path)
wm_img = cv2.imread(str(wm_path), cv2.IMREAD_GRAYSCALE) if wm_path.exists() else None

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
if host_img is not None:
    plt.imshow(host_img)
    plt.title(f'Host: {host_path.name}')
else:
    plt.text(0.5, 0.5, 'Cannot read host image', ha='center', va='center')
plt.axis('off')

plt.subplot(1, 2, 2)
if wm_img is not None:
    plt.imshow(wm_img, cmap='gray')
    plt.title('Watermark: cict.png')
else:
    plt.text(0.5, 0.5, 'Watermark not found', ha='center', va='center')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
if not attack_dir.exists() or not recover_dir.exists():
    print('Chua co ket qua cho host da chon. Hay chay scripts/main.py truoc.')
else:
    attacks = sorted([p for p in attack_dir.glob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp'}])
    recovers = sorted([p for p in recover_dir.glob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp'}])
    print('So attacked images:', len(attacks))
    print('So recovered images:', len(recovers))

    show_attacks = attacks[:3]
    show_recovers = recovers[:3]

    n = max(len(show_attacks), len(show_recovers), 1)
    plt.figure(figsize=(10, 3 * n))

    for i in range(n):
        plt.subplot(n, 2, i * 2 + 1)
        if i < len(show_attacks):
            img = read_rgb(show_attacks[i])
            if img is not None:
                plt.imshow(img)
                plt.title(f'Attack: {show_attacks[i].name}')
            else:
                plt.text(0.5, 0.5, 'Cannot read attack image', ha='center', va='center')
        else:
            plt.text(0.5, 0.5, 'No attack image', ha='center', va='center')
        plt.axis('off')

        plt.subplot(n, 2, i * 2 + 2)
        if i < len(show_recovers):
            img = cv2.imread(str(show_recovers[i]), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                plt.imshow(img, cmap='gray')
                plt.title(f'Recover: {show_recovers[i].name}')
            else:
                plt.text(0.5, 0.5, 'Cannot read recover image', ha='center', va='center')
        else:
            plt.text(0.5, 0.5, 'No recover image', ha='center', va='center')
        plt.axis('off')

    plt.tight_layout()
    plt.show()